# Quantum Phase Estimation, Order Finding y Shor

Este notebook acompaña el módulo **QPE, Order Finding y Shor**. Está escrito como material autoexplicativo para estudiar la conexión entre estimación de fase, periodicidad modular y factorización entera. La notación de Dirac se escribe sin macros personalizadas para preservar compatibilidad con JupyterLab, Anaconda y Google Colab; por ejemplo, usamos

$$
\left|u\right\rangle, \qquad \left\langle u\right|, \qquad U\left|u\right\rangle=e^{2\pi i\varphi}\left|u\right\rangle.
$$

El objetivo conceptual es entender cómo una fase cuántica, que no se observa directamente, puede convertirse en una fracción racional y luego en el orden multiplicativo que permite factorizar enteros compuestos.

## Objetivos de aprendizaje

Al finalizar el notebook podrás explicar qué estima QPE, por qué el registro de conteo codifica una fase binaria, cómo la inverse QFT transforma una onda de fase en una distribución de medición, por qué el operador de multiplicación modular tiene fases de la forma $s/r$, cómo las fracciones continuas permiten reconstruir candidatos para $r$, y cómo Shor convierte un orden útil en factores no triviales de $N$.

El notebook no repite la introducción básica a números complejos, qubits o QFT. Esos conceptos se usan como prerrequisitos y se retoman únicamente cuando son necesarios para interpretar QPE, order finding y Shor.

In [ ]:
import math
from math import gcd, pi
from fractions import Fraction
from typing import List, Tuple, Optional, Dict

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

## 1. Qué significa estimar una fase

Sea $U$ un operador unitario y sea $\left|u\right\rangle$ un eigenvector:

$$
U\left|u\right\rangle=e^{2\pi i\varphi}\left|u\right\rangle,
\qquad \varphi\in[0,1).
$$

QPE intenta estimar $\varphi$. La fase se expresa normalizada: $\varphi=1/2$ significa una rotación angular de $\pi$, mientras que $\varphi=5/8$ significa una rotación de $5\pi/4$. La clave es que aplicar $U^k$ al eigenvector produce

$$
U^k\left|u\right\rangle=e^{2\pi i k\varphi}\left|u\right\rangle.
$$

Por eso las potencias controladas de $U$ permiten escribir múltiplos de la fase en el registro de conteo.

In [ ]:
def plot_phase_on_unit_circle(phi: float, title: str = "Fase como punto en el círculo unitario") -> None:
    """Visualiza e^{2πiφ} en el círculo unitario."""
    theta = 2 * np.pi * phi
    fig, ax = plt.subplots(figsize=(5, 5))
    t = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(t), np.sin(t))
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.arrow(0, 0, np.cos(theta), np.sin(theta), head_width=0.06, length_includes_head=True)
    ax.scatter([np.cos(theta)], [np.sin(theta)])
    ax.text(np.cos(theta)*1.12, np.sin(theta)*1.12, f"φ={phi:.4f}")
    ax.set_aspect("equal")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.set_title(title)
    plt.show()

plot_phase_on_unit_circle(5/8, "Ejemplo: fase estimada 5/8")

## 2. Fases binarias

Con $t$ qubits de conteo se distinguen $2^t$ posiciones en el intervalo $[0,1)$. Una medición $m$ se interpreta como

$$
\widetilde{\varphi}=\frac{m}{2^t}.
$$

Si $t=3$ y la cadena observada es $101_2$, entonces $m=5$ y la estimación es $5/8$. Esta interpretación será esencial cuando el resultado de QPE deba convertirse en una fracción $s/r$.

In [ ]:
def binary_phase_table(t: int) -> None:
    """Muestra las fases representables con t qubits de conteo."""
    rows = []
    for m in range(2**t):
        rows.append((format(m, f"0{t}b"), m, Fraction(m, 2**t)))
    for bits, m, frac in rows:
        print(f"{bits} -> {m:2d}/{2**t} = {frac}")

binary_phase_table(3)

In [ ]:
def plot_binary_phase_grid(t: int, highlight: Optional[int] = None) -> None:
    """Visualiza la malla binaria de fases en [0,1)."""
    xs = np.arange(2**t) / (2**t)
    ys = np.zeros_like(xs)
    plt.figure(figsize=(9, 1.8))
    plt.scatter(xs, ys)
    for m, x in enumerate(xs):
        plt.text(x, 0.05, format(m, f"0{t}b"), ha="center", fontsize=8, rotation=45)
    if highlight is not None:
        plt.scatter([highlight/2**t], [0], s=180)
        plt.text(highlight/2**t, -0.12, f"m={highlight}", ha="center")
    plt.yticks([])
    plt.xlabel("fase normalizada")
    plt.title(f"Malla de fases con t={t} qubits")
    plt.xlim(-0.03, 1.03)
    plt.show()

plot_binary_phase_grid(3, highlight=5)

## 3. Distribución ideal de QPE

Cuando la fase exacta no cae en la malla binaria, QPE no devuelve siempre un único entero. La distribución de probabilidad se concentra alrededor de los enteros $m$ cercanos a $2^t\varphi$. Una forma útil de visualizar la distribución ideal es construir el estado del registro de conteo antes de la inverse QFT y aplicar la transformada inversa discreta.

In [ ]:
def qft_matrix(M: int, inverse: bool = False) -> np.ndarray:
    """Matriz QFT de dimensión M, con normalización unitaria."""
    omega = np.exp(2j * np.pi / M)
    j, k = np.meshgrid(np.arange(M), np.arange(M), indexing="ij")
    F = omega ** (j * k) / np.sqrt(M)
    return F.conj().T if inverse else F


def qpe_distribution(phi: float, t: int) -> np.ndarray:
    """Distribución ideal de QPE para una fase φ usando t qubits de conteo."""
    M = 2**t
    k = np.arange(M)
    state_before_iqft = np.exp(2j * np.pi * phi * k) / np.sqrt(M)
    state_after_iqft = qft_matrix(M, inverse=True) @ state_before_iqft
    return np.abs(state_after_iqft) ** 2


def plot_qpe_distribution(phi: float, t: int) -> None:
    probs = qpe_distribution(phi, t)
    labels = [format(i, f"0{t}b") for i in range(2**t)]
    plt.figure(figsize=(9, 4))
    plt.bar(range(2**t), probs)
    plt.xticks(range(2**t), labels, rotation=90)
    plt.ylabel("probabilidad")
    plt.xlabel("resultado del registro de conteo")
    plt.title(f"Distribución ideal de QPE para φ={phi:.5f}, t={t}")
    plt.tight_layout()
    plt.show()

plot_qpe_distribution(5/8, 3)
plot_qpe_distribution(2/7, 4)

## 4. Diagrama conceptual de QPE

El diagrama siguiente no intenta reemplazar el circuito real. Su función es mostrar el flujo de información: el registro de conteo se prepara en superposición, las potencias controladas escriben fases y la inverse QFT convierte esas fases en una cadena medible.

In [ ]:
def draw_qpe_diagram() -> None:
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.axis("off")
    boxes = [
        (0.05, 0.62, "conteo\n|0...0>"),
        (0.25, 0.62, "H en\nconteo"),
        (0.47, 0.62, "U^{2^j}\ncontroladas"),
        (0.70, 0.62, "inverse\nQFT"),
        (0.88, 0.62, "medición\nm"),
        (0.47, 0.18, "trabajo\n|u>")
    ]
    for x, y, txt in boxes:
        ax.text(x, y, txt, ha="center", va="center", bbox=dict(boxstyle="round", fc="white", ec="black"))
    for x1, x2 in [(0.12,0.20),(0.31,0.40),(0.55,0.62),(0.76,0.82)]:
        ax.annotate("", xy=(x2,0.62), xytext=(x1,0.62), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.47,0.48), xytext=(0.47,0.28), arrowprops=dict(arrowstyle="->"))
    ax.set_title("Arquitectura conceptual de Quantum Phase Estimation")
    plt.show()

draw_qpe_diagram()

## 5. Potencias controladas

La razón por la que aparecen potencias $U^{2^j}$ es binaria. El qubit $j$ del registro de conteo decide si la rama correspondiente acumula una fase multiplicada por $2^j$. En una implementación, la sintaxis exacta depende del marco de software, pero la intención matemática es siempre controlar $U^{2^j}$.

In [ ]:
def controlled_power_table(t: int) -> None:
    for j in range(t):
        print(f"qubit de conteo {j}: controla U^{2**j}")

controlled_power_table(6)

## 6. Order finding

Dado $x$ coprimo con $N$, el orden multiplicativo de $x$ módulo $N$ es el menor entero positivo $r$ tal que

$$
x^r\equiv 1 \pmod N.
$$

La sucesión $x^a\bmod N$ es periódica con período $r$. Shor usa QPE porque esa periodicidad puede representarse como fase de un operador unitario de multiplicación modular.

In [ ]:
def order_mod(x: int, N: int, limit: Optional[int] = None) -> Optional[int]:
    """Devuelve el menor r>0 tal que x^r = 1 mod N, si gcd(x,N)=1."""
    if gcd(x, N) != 1:
        return None
    if limit is None:
        limit = 4 * N
    value = 1
    for r in range(1, limit + 1):
        value = (value * x) % N
        if value == 1:
            return r
    return None

print("orden de 4 módulo 81:", order_mod(4, 81))
print("orden de 2 módulo 15:", order_mod(2, 15))
print("orden de 2 módulo 21:", order_mod(2, 21))

In [ ]:
def modular_sequence(x: int, N: int, length: int) -> List[int]:
    return [pow(x, a, N) for a in range(length)]


def plot_modular_periodicity(x: int, N: int, length: int = 40) -> None:
    seq = modular_sequence(x, N, length)
    r = order_mod(x, N)
    plt.figure(figsize=(10, 4))
    plt.plot(range(length), seq, marker="o")
    if r is not None:
        for k in range(0, length, r):
            plt.axvline(k, linestyle="--", alpha=0.4)
    plt.xlabel("exponente a")
    plt.ylabel(f"{x}^a mod {N}")
    plt.title(f"Periodicidad modular: x={x}, N={N}, orden r={r}")
    plt.show()

plot_modular_periodicity(2, 15, 24)
plot_modular_periodicity(4, 81, 60)

## 7. De periodicidad a fase

El operador

$$
U_x\left|y\right\rangle=\left|xy\bmod N\right\rangle
$$

es una permutación reversible cuando $\gcd(x,N)=1$. Su estructura cíclica implica que sus eigenvalores relevantes tienen forma

$$
e^{2\pi i s/r}, \qquad s=0,1,\ldots,r-1.
$$

Por tanto, QPE aplicada a $U_x$ produce aproximaciones a fracciones $s/r$.

In [ ]:
def plot_order_phases(r: int) -> None:
    """Visualiza las fases s/r como puntos en el círculo unitario."""
    s_vals = np.arange(r)
    angles = 2*np.pi*s_vals/r
    fig, ax = plt.subplots(figsize=(5, 5))
    t = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(t), np.sin(t))
    ax.scatter(np.cos(angles), np.sin(angles))
    for s, a in zip(s_vals, angles):
        ax.text(1.15*np.cos(a), 1.15*np.sin(a), f"{s}/{r}", ha="center", va="center", fontsize=8)
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_aspect("equal")
    ax.set_title(f"Fases e^(2π i s/{r}) asociadas a un ciclo de longitud {r}")
    plt.show()

plot_order_phases(4)
plot_order_phases(7)

## 8. Picos esperados tras la inverse QFT

Si el período es $r$ y el registro de conteo tiene tamaño $M=2^t$, las frecuencias probables se ubican cerca de

$$
k_s \approx \frac{sM}{r}, \qquad s=0,1,\ldots,r-1.
$$

La medición produce enteros cercanos a esos valores. Después se usa $k/M$ como aproximación de $s/r$.

In [ ]:
def expected_peaks(M: int, r: int) -> List[int]:
    """Enteros cercanos a sM/r."""
    return sorted(set(int(round(s * M / r)) for s in range(r)))

for r in [4, 6, 27]:
    print(f"M=512, r={r}: picos aproximados =", expected_peaks(512, r)[:20])

In [ ]:
def plot_expected_peaks(M: int, r: int) -> None:
    peaks = expected_peaks(M, r)
    y = np.zeros(M)
    for p in peaks:
        if 0 <= p < M:
            y[p] = 1
    plt.figure(figsize=(10, 2.8))
    plt.stem(range(M), y, basefmt=" ")
    plt.xlabel("k")
    plt.ylabel("pico ideal")
    plt.title(f"Ubicación aproximada de picos para M={M}, r={r}")
    plt.xlim(0, M)
    plt.show()

plot_expected_peaks(128, 4)
plot_expected_peaks(128, 6)

## 9. Fracciones continuas

QPE devuelve una fracción binaria $m/2^t$. Para extraer el período buscamos una fracción $s/r$ cercana. Las fracciones continuas construyen convergentes que son candidatos racionales especialmente buenos.

In [ ]:
def continued_fraction(x: float, max_terms: int = 12) -> List[int]:
    terms = []
    y = x
    for _ in range(max_terms):
        a = math.floor(y)
        terms.append(a)
        frac = y - a
        if abs(frac) < 1e-12:
            break
        y = 1 / frac
    return terms


def convergents_from_cf(cf: List[int]) -> List[Fraction]:
    convs = []
    for i in range(1, len(cf) + 1):
        f = Fraction(cf[i-1], 1)
        for a in reversed(cf[:i-1]):
            f = a + Fraction(1, f)
        convs.append(f)
    return convs

cf = [1, 4, 2, 1]
print("fracción continua:", cf)
print("convergentes:", convergents_from_cf(cf))

In [ ]:
def plot_convergents(x: float, max_terms: int = 10) -> None:
    cf = continued_fraction(x, max_terms)
    convs = convergents_from_cf(cf)
    errors = [abs(float(c) - x) for c in convs]
    labels = [f"{c.numerator}/{c.denominator}" for c in convs]
    plt.figure(figsize=(9, 4))
    plt.semilogy(range(1, len(errors)+1), errors, marker="o")
    plt.xticks(range(1, len(errors)+1), labels, rotation=45)
    plt.ylabel("error absoluto")
    plt.xlabel("convergente")
    plt.title(f"Convergentes de {x:.6f}")
    plt.tight_layout()
    plt.show()

plot_convergents(11/9, max_terms=6)
plot_convergents(2/7, max_terms=8)

## 10. Reconstrucción de un candidato de orden

A partir de una medición $m$, se forma $m/M$. Los denominadores de los convergentes son candidatos para $r$. Cada candidato debe verificarse mediante la congruencia

$$
x^r\equiv1\pmod N.
$$

El algoritmo no acepta un denominador únicamente porque aparece en una aproximación; lo prueba contra la aritmética modular.

In [ ]:
def order_candidates_from_measurement(m: int, M: int, max_denominator: int = 100) -> List[Fraction]:
    x = m / M
    cf = continued_fraction(x, max_terms=20)
    convs = convergents_from_cf(cf)
    return [c for c in convs if c.denominator <= max_denominator]


def verify_order_candidates(x: int, N: int, candidates: List[Fraction]) -> List[Tuple[Fraction, bool]]:
    results = []
    for frac in candidates:
        r = frac.denominator
        results.append((frac, pow(x, r, N) == 1))
    return results

m = 85
M = 512
cands = order_candidates_from_measurement(m, M, max_denominator=30)
print("candidatos desde 85/512:", cands)
print("verificación para x=2, N=15:", verify_order_candidates(2, 15, cands))

## 11. Flujo completo de Shor

Shor combina piezas clásicas y cuánticas. El subproblema cuántico es order finding; el resto del algoritmo interpreta el orden para obtener factores.

In [ ]:
def shor_classical_postprocess(N: int, x: int, r: int) -> Optional[Tuple[int, int]]:
    """Intenta obtener factores no triviales de N a partir de x y r."""
    if r is None or r % 2 == 1:
        return None
    y = pow(x, r // 2, N)
    if y == N - 1:
        return None
    p = gcd(y - 1, N)
    q = gcd(y + 1, N)
    if p in (1, N) or q in (1, N):
        return None
    return p, q

for N, x in [(15, 2), (21, 2), (35, 3)]:
    r = order_mod(x, N)
    print(f"N={N}, x={x}, r={r}, factores={shor_classical_postprocess(N, x, r)}")

In [ ]:
def draw_shor_flow() -> None:
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.axis("off")
    steps = [
        "elegir x", "gcd(x,N)", "QPE sobre U_x", "fracciones\ncontinuas", "candidato r", "gcd(x^{r/2}±1,N)", "factores"
    ]
    xs = np.linspace(0.05, 0.95, len(steps))
    for x, label in zip(xs, steps):
        ax.text(x, 0.5, label, ha="center", va="center", bbox=dict(boxstyle="round", fc="white", ec="black"))
    for x1, x2 in zip(xs[:-1], xs[1:]):
        ax.annotate("", xy=(x2-0.045, 0.5), xytext=(x1+0.045, 0.5), arrowprops=dict(arrowstyle="->"))
    ax.set_title("Flujo conceptual de Shor")
    plt.show()

draw_shor_flow()

## 12. Precisión frente a número de qubits

Cada qubit adicional en el registro de conteo duplica la resolución de fase. Sin embargo, también incrementa el número de potencias controladas y la profundidad del circuito. La siguiente gráfica muestra la escala de cuantización $1/2^t$.

In [ ]:
def plot_precision_vs_qubits(tmax: int = 14) -> None:
    ts = np.arange(1, tmax + 1)
    err = 1 / (2 ** ts)
    plt.figure(figsize=(8, 4))
    plt.semilogy(ts, err, marker="o")
    plt.xlabel("qubits de conteo t")
    plt.ylabel("tamaño de celda 1/2^t")
    plt.title("Resolución binaria de fase")
    plt.xticks(ts)
    plt.show()

plot_precision_vs_qubits()

## 13. Distribuciones aproximadas para distintas precisiones

La misma fase no exacta se representa mejor conforme aumenta $t$. La gráfica siguiente compara la distribución de QPE para $\varphi=2/7$ con distintos tamaños de registro.

In [ ]:
def compare_qpe_precisions(phi: float, t_values: List[int]) -> None:
    for t in t_values:
        probs = qpe_distribution(phi, t)
        plt.figure(figsize=(8, 3))
        plt.bar(range(2**t), probs)
        plt.xlabel("m")
        plt.ylabel("probabilidad")
        plt.title(f"QPE para φ={phi:.5f}, t={t}")
        plt.tight_layout()
        plt.show()

compare_qpe_precisions(2/7, [3, 4, 5])

## 14. Implementación opcional con Qiskit

Las celdas siguientes intentan importar Qiskit. Si el paquete no está disponible y el entorno permite instalación, se intentará instalarlo. Si no es posible, el notebook continúa con las visualizaciones numéricas y deja la sección de circuito como esquema conceptual.

### Nota de compatibilidad para diagramas de circuitos

El dibujador Matplotlib de Qiskit requiere la dependencia opcional `pylatexenc`. Para que la opción **Ejecutar todas las celdas** sea estable en JupyterLab, Anaconda y Google Colab, el notebook intenta instalar esa dependencia cuando sea necesario. Si el entorno no permite instalar paquetes, el circuito se muestra automáticamente en formato de texto y la ejecución continúa sin detenerse.

In [ ]:
import importlib
import subprocess
import sys
from IPython.display import display


def try_import_or_install(import_name: str, pip_name: Optional[str] = None):
    """Intenta importar un paquete; si falla, intenta instalarlo. Devuelve el módulo o None.

    La función está pensada para JupyterLab, Anaconda y Google Colab. Si el entorno no tiene acceso a
    instalación por pip, el notebook continúa ejecutándose y usa rutas numéricas o representaciones de texto.
    """
    try:
        return importlib.import_module(import_name)
    except Exception:
        pass

    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name, "-q"])
        return importlib.import_module(import_name)
    except Exception as exc:
        print(f"No se pudo importar o instalar {import_name}: {exc}")
        return None


def display_qiskit_circuit(qc):
    """Muestra un circuito de Qiskit sin detener la ejecución si falta pylatexenc.

    Qiskit usa `pylatexenc` para `qc.draw("mpl")`. Si esa dependencia opcional no está instalada,
    se intenta instalar. Si aun así no está disponible, se muestra el circuito en modo texto para que
    `Ejecutar todas las celdas` no falle.
    """
    try:
        display(qc.draw(output="mpl"))
        return
    except Exception as exc:
        message = str(exc)
        if "pylatexenc" in message or "MatplotlibDrawer" in message:
            pylatexenc = try_import_or_install("pylatexenc")
            if pylatexenc is not None:
                try:
                    display(qc.draw(output="mpl"))
                    return
                except Exception as exc_retry:
                    print(f"No se pudo usar el dibujador Matplotlib después de instalar pylatexenc: {exc_retry}")
        else:
            print(f"No se pudo usar el dibujador Matplotlib de Qiskit: {exc}")

    print("Se muestra el circuito en formato de texto para mantener la ejecución completa del notebook.")
    print(qc.draw(output="text"))


qiskit = try_import_or_install("qiskit")
qiskit_aer = try_import_or_install("qiskit_aer", "qiskit-aer")
HAS_QISKIT = qiskit is not None
HAS_QISKIT_AER = qiskit_aer is not None
print("Qiskit disponible:", HAS_QISKIT)
print("Qiskit Aer disponible:", HAS_QISKIT_AER)


In [ ]:
if HAS_QISKIT:
    from qiskit import QuantumCircuit, transpile
    if HAS_QISKIT_AER:
        from qiskit_aer import AerSimulator

    def inverse_qft_qiskit(qc: QuantumCircuit, qubits: List[int]) -> None:
        """Inverse QFT simple sobre una lista de qubits."""
        n = len(qubits)
        for i in range(n // 2):
            qc.swap(qubits[i], qubits[n - 1 - i])
        for j in range(n):
            for m in range(j):
                qc.cp(-np.pi / (2 ** (j - m)), qubits[m], qubits[j])
            qc.h(qubits[j])

    def qpe_phase_gate_circuit(phi: float, t: int) -> QuantumCircuit:
        """QPE para una compuerta U que añade fase e^(2πiφ) al estado |1> del trabajo."""
        qc = QuantumCircuit(t + 1, t)
        work = t
        qc.x(work)  # prepara |1>, eigenestado de una compuerta de fase
        qc.h(range(t))
        for j in range(t):
            angle = 2 * np.pi * phi * (2 ** j)
            qc.cp(angle, j, work)
        inverse_qft_qiskit(qc, list(range(t)))
        qc.measure(range(t), range(t))
        return qc

    qc = qpe_phase_gate_circuit(5/8, 3)
    display_qiskit_circuit(qc)
else:
    print("Se omite el circuito Qiskit porque el paquete no está disponible en este entorno.")


In [ ]:
if HAS_QISKIT and HAS_QISKIT_AER:
    sim = AerSimulator()
    qc = qpe_phase_gate_circuit(5/8, 3)
    tqc = transpile(qc, sim)
    result = sim.run(tqc, shots=2048).result()
    counts = result.get_counts()
    print(counts)

    plt.figure(figsize=(6, 3))
    plt.bar(counts.keys(), counts.values())
    plt.xlabel("resultado")
    plt.ylabel("conteos")
    plt.title("QPE exacta para φ=5/8")
    plt.show()
elif HAS_QISKIT and not HAS_QISKIT_AER:
    print("Qiskit está disponible, pero Qiskit Aer no. Se muestra la distribución teórica equivalente.")
    plot_qpe_distribution(5/8, 3)
else:
    print("Distribución teórica equivalente:")
    plot_qpe_distribution(5/8, 3)


## 15. Implementación opcional con Cirq

La sección siguiente construye un esquema de QPE en Cirq cuando el paquete está disponible. Si el entorno no permite instalarlo, las celdas informan la omisión sin detener el notebook.

In [ ]:
cirq = try_import_or_install("cirq")
HAS_CIRQ = cirq is not None
print("Cirq disponible:", HAS_CIRQ)

In [ ]:
if HAS_CIRQ:
    def qpe_cirq_phase_gate(phi: float, t: int):
        qubits = cirq.LineQubit.range(t + 1)
        work = qubits[-1]
        circuit = cirq.Circuit()
        circuit.append(cirq.X(work))
        circuit.append(cirq.H.on_each(*qubits[:t]))
        for j in range(t):
            # CZPowGate con exponent e produce fase exp(i*pi*e) sobre |11>.
            exponent = 2 * phi * (2 ** j)
            circuit.append(cirq.CZPowGate(exponent=exponent).on(qubits[j], work))
        circuit.append(cirq.measure(*qubits[:t], key="m"))
        return circuit

    c = qpe_cirq_phase_gate(5/8, 3)
    print(c)
else:
    print("Se omite el circuito Cirq porque el paquete no está disponible en este entorno.")

## 16. Ejemplo paso a paso: de una medición a un factor

Supongamos que la subrutina cuántica de order finding para $N=15$ y $x=2$ proporciona evidencia de que el orden es $r=4$. El paso final es clásico:

$$
2^{r/2}=2^2=4,\qquad
\gcd(4-1,15)=3,\qquad
\gcd(4+1,15)=5.
$$

La factorización aparece porque $2^4\equiv1\pmod{15}$ implica que $2^4-1$ es múltiplo de $15$, y la factorización algebraica de $2^4-1$ contiene los divisores no triviales.

In [ ]:
def explain_factor_step(N: int, x: int) -> None:
    r = order_mod(x, N)
    print(f"N={N}, x={x}, orden r={r}")
    if r is None:
        print("x no es coprimo con N; gcd ya da un factor.")
        print("gcd(x,N)=", gcd(x, N))
        return
    factors = shor_classical_postprocess(N, x, r)
    print("factores desde r:", factors)

explain_factor_step(15, 2)
explain_factor_step(21, 2)

## 17. Ejercicios guiados

**Ejercicio 1.** Un registro de conteo de cuatro qubits produce la cadena $0110_2$. Interpreta la estimación de fase.  
**Desarrollo.** La cadena representa $6$ y la escala es $16$, por lo que $\widetilde{\varphi}=6/16=3/8$.

**Ejercicio 2.** Para $x=4$ y $N=81$, verifica por código que el orden es $27$.  
**Desarrollo.** La función `order_mod(4, 81)` recorre potencias sucesivas hasta encontrar el primer regreso a $1$.

**Ejercicio 3.** Si el candidato de orden es impar, explica por qué no se puede extraer factor mediante $x^{r/2}\pm1$.  
**Desarrollo.** El exponente $r/2$ no es entero, por lo que la identidad usada en el paso de factorización no se aplica.

In [ ]:
print("Ejercicio 1: 0110 con 4 qubits ->", Fraction(int("0110", 2), 2**4))
print("Ejercicio 2: orden de 4 módulo 81 ->", order_mod(4, 81))
print("Ejercicio 3: r=27 es impar ->", "repetir con otra elección" if 27 % 2 else "puede continuar")

## 18. Ejercicios propuestos

1. Construye la tabla de $2^a\bmod 21$ para $a=0,1,\ldots,12$ y localiza el período.
2. Grafica la distribución ideal de QPE para $\varphi=3/10$ usando $t=4,5,6$.
3. Usa fracciones continuas para aproximar $85/512$ y lista los denominadores candidatos menores que $20$.
4. Para $N=35$ y $x=3$, calcula el orden y verifica si permite obtener factores.
5. Explica en un párrafo por qué la salida de QPE en order finding debe interpretarse como aproximación a $s/r$ y no como el orden directamente.

## 19. Resumen final

Quantum Phase Estimation extrae una fase asociada a un eigenvalor unitario. Order finding convierte la periodicidad modular en un problema espectral, de modo que las fases medidas tienen forma $s/r$. Las fracciones continuas reconstruyen candidatos racionales a partir de la estimación binaria. Shor combina estas piezas para convertir un orden útil en factores no triviales mediante máximos comunes divisores.

El mensaje central es que la factorización se vuelve accesible al transformar un problema aritmético en un problema de interferencia y luego regresar al lenguaje de enteros mediante post-procesamiento clásico.